# Day 3.3 — Transparent Persistent Memory

## Before you begin

### Learning outcomes

- Run the full memory lifecycle: add, search, update, delete.
- Prove that memory is scoped per user, so one student never sees another's records.
- Watch keyword retrieval fail on conflicting records and apply an explicit resolution rule.

Architecture reference: [Day 3 diagrams D10](../diagrams/source/day_03.md).

### Expected observation

Asha's two seeded preferences are found by search, Omar's store stays empty, an updated record shows a new `updated_at`, and the deleted record disappears. IDs and timestamps differ on every run.

## Concept briefing

## What deserves persistent memory

Saving every sentence creates a surveillance log, not useful memory. A memory record
should be useful, appropriately scoped, attributable and controllable by the user. At a
minimum, students should be able to inspect, correct and delete records.

Useful metadata includes user identity, source, creation time, update time and possibly
expiry. Conflicting memories require a policy: prefer confirmed newer information, ask
the user, or preserve both with provenance. Similarity alone cannot decide truth.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/safe_task_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Step 1 — Load the synthetic users

`data/synthetic_users.json` holds two fictional students. We never put real people, real
preferences, or real messages into a memory store built in class.

In [ ]:
import json

users = json.loads((PROJECT_ROOT / "data" / "synthetic_users.json").read_text(encoding="utf-8"))
for user in users:
    print(user["user_id"], "->", user["preferences"])

## Step 2 — Create the store and seed one user

`SQLiteMemoryStore()` with no path keeps the database in RAM, which is perfect for a lesson.
Every record carries who it belongs to, where it came from, and when it was written.

In [ ]:
from safe_task_agent import SQLiteMemoryStore

store = SQLiteMemoryStore()          # in-memory SQLite; nothing is written to disk
asha, omar = users[0]["user_id"], users[1]["user_id"]

for preference in users[0]["preferences"]:
    store.add(asha, preference, source="synthetic_dataset")

for record in store.all(asha):
    print("id       :", record.id[:8], "...")
    print("text     :", record.text)
    print("source   :", record.source, "| created:", record.created_at)
    print()

## Step 3 — Retrieve only what is relevant

An agent should not paste every stored memory into the prompt. It retrieves the few records
that match the current request. Our matcher is deliberately simple word overlap so you can
read it in `src/safe_task_agent/memory.py`.

In [ ]:
for query in ["what time should we meet?", "how should I write the email?", "which lab bench?"]:
    hits = store.search(asha, query)
    print(f"query: {query!r}")
    print("  ->", [record.text for record in hits] or "no matching memory")

## Step 4 — Memory is scoped to one user

Every query carries a `user_id`. Omar cannot see Asha's records, and there is no code path in
the store that ignores the scope.

In [ ]:
print("Asha's records :", len(store.all(asha)))
print("Omar's records :", len(store.all(omar)))
print("Omar searching for Asha's preference:", store.search(omar, "meetings after 10:00"))

## Step 5 — The user can correct and delete

A memory the user cannot fix or remove is a liability, not a feature.

In [ ]:
first = store.all(asha)[-1]           # the oldest of Asha's records
print("Original :", first.text)

updated = store.update(asha, first.id, "Prefer meetings after 11:00")
print("Corrected:", updated.text)
print("created_at == updated_at?", updated.created_at == updated.updated_at)

print("Deleted  :", store.delete(asha, updated.id))
print("Remaining:", [r.text for r in store.all(asha)])

## Step 6 — Break it: two memories that contradict each other

Now save a preference that conflicts with one already stored, and search for a meeting time.

In [ ]:
# We keep a handle on each record so we know which statement came first.
older = store.add(asha, "Prefer meetings after 10:00", source="explicit_user_statement")
newer = store.add(asha, "Never schedule meetings before 14:00", source="explicit_user_statement")

hits = store.search(asha, "when should we schedule meetings?")
print("Records returned for one question:")
for record in hits:
    print(" -", record.text, "| written at", record.created_at)

print("\nBoth match the words in the question. Word overlap ranks text similarity;")
print("it has no idea that these two sentences cannot both be obeyed.")

## Step 7 — Resolve the conflict with a rule, not a guess

The store cannot decide truth, so the application must state a policy. Ours: *the newest
explicitly confirmed statement wins, and the older one is removed so it can never be
retrieved again.*

In [ ]:
print("Stated first :", older.text)
print("Stated later :", newer.text)
print("Rule         : the most recent explicit statement wins")

print("\nRemoving the superseded record:", store.delete(asha, older.id))
print("Memory now answers with one voice:",
      [r.text for r in store.search(asha, "when should we schedule meetings?")])

print("\nWe used the order our application recorded, not the timestamps: two writes")
print("in the same millisecond can carry the same updated_at, so a timestamp alone")
print("is not always enough to say which statement is newer.")

### Try it yourself

Predict whether records survive if you close and reopen a *file-backed* store. Run the worked
solution to find out. (It writes to a temporary folder, so nothing lands in the course repo.)

In [ ]:
# --- Worked solution ---
import tempfile

db_path = Path(tempfile.mkdtemp()) / "memory.db"       # a throwaway folder

first_session = SQLiteMemoryStore(db_path)             # same class, now backed by a file
first_session.add("fictional_asha", "Prefer meetings after 11:00", "explicit_user_statement")
first_session.connection.close()                       # end of "session one"

second_session = SQLiteMemoryStore(db_path)            # a brand new process would do this
print("Database file    :", db_path.name)
print("Records after reopening:", [r.text for r in second_session.all("fictional_asha")])

# Persistence is a storage choice, not a model capability: the same class, one argument
# different. Nothing about the model changed.

### Checkpoint

**1. Why does every method of the store take a `user_id`?**

<details><summary>Show answer</summary>

So retrieval is isolated per user by construction. If the scope were optional, one forgotten argument would leak one student's preferences into another student's prompt.

</details>

**2. The store returned two contradictory memories. Whose job is it to fix that?**

<details><summary>Show answer</summary>

The application's. Search ranks by similarity, and similarity cannot decide which statement is true. We applied an explicit rule - newest confirmed statement wins, older record deleted. Other valid rules: ask the user, or keep both with provenance and let the user choose.

</details>

### Recap

- **Limitation we saw:** Keyword retrieval happily returned two memories that contradicted each other.
- **Layer we added:** A user-scoped store with a full lifecycle (add, search, update, delete) plus a written conflict-resolution rule.
- **Evidence it worked:** Omar's store stayed empty, the corrected record changed text, the deleted record disappeared, and after resolution the conflicting query returned a single answer.